# HMoRA Dense vs Sparse Study

This notebook analyzes artifacts produced by `scripts/local/hmora_sparse_study.py`.

The benchmark script is the source of truth. This notebook only reads result files and plots comparisons.

## Suggested command

```bash
python3 scripts/local/hmora_sparse_study.py \
  --tier both \
  --pass-mode both \
  --hidden-sizes 512,1024,2048 \
  --ranks 4,8 \
  --num-experts-values 2,4,8,16,32 \
  --top-k-values 1,2,4,8,16,32 \
  --batch-size-values 2,4 \
  --seq-len-values 128,512,1024 \
  --use-hydra-lora true \
  --share-router-qkv true \
  --share-router-wi true
```

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
RESULTS_PATH = Path('outputs/hmora_sparse_study/results.jsonl')
assert RESULTS_PATH.exists(), f'Missing results file: {RESULTS_PATH}'

rows = [json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
df = pd.DataFrame(rows)
df.head()

In [ ]:
dense = df[df['variant'] == 'dense_packed'].copy()
sparse = df[df['variant'] == 'true_sparse'].copy()

merge_keys = [
    'tier', 'pass_mode', 'hidden_size', 'intermediate_size', 'rank', 'num_experts',
    'top_k', 'batch_size', 'seq_len', 'use_hydra_lora', 'share_router_qkv', 'share_router_wi'
]

cmp = dense.merge(
    sparse,
    on=merge_keys,
    suffixes=('_dense', '_sparse'),
)
cmp['speedup_sparse_vs_dense'] = cmp['mean_step_seconds_dense'] / cmp['mean_step_seconds_sparse']
cmp['tokens_per_second_gain'] = cmp['tokens_per_second_sparse'] / cmp['tokens_per_second_dense']
cmp[['tier', 'pass_mode', 'num_experts', 'top_k', 'speedup_sparse_vs_dense', 'tokens_per_second_gain']].head()

In [ ]:
summary = (
    cmp.groupby(['tier', 'pass_mode', 'num_experts', 'top_k'])[['speedup_sparse_vs_dense', 'tokens_per_second_gain']]
    .mean()
    .reset_index()
    .sort_values(['tier', 'pass_mode', 'num_experts', 'top_k'])
)
summary.head(20)

In [ ]:
for tier in sorted(summary['tier'].unique()):
    tier_df = summary[(summary['tier'] == tier) & (summary['pass_mode'] == 'forward_backward')]
    if tier_df.empty:
        continue
    plt.figure(figsize=(8, 5))
    for num_experts, group in tier_df.groupby('num_experts'):
        group = group.sort_values('top_k')
        plt.plot(group['top_k'], group['speedup_sparse_vs_dense'], marker='o', label=f'E={num_experts}')
    plt.axhline(1.0, color='black', linestyle='--', linewidth=1)
    plt.title(f'{tier}: sparse speedup over dense (forward+backward)')
    plt.xlabel('top_k')
    plt.ylabel('dense step time / sparse step time')
    plt.legend()
    plt.show()

In [ ]:
quality_cols = [
    'tier', 'pass_mode', 'num_experts', 'top_k',
    'equiv_pass_dense', 'equiv_output_max_abs_diff_dense',
    'equiv_input_grad_max_abs_diff_dense', 'equiv_param_grad_max_abs_diff_dense'
]
available_quality_cols = [col for col in quality_cols if col in cmp.columns]
cmp[available_quality_cols].drop_duplicates().head(20)